# Curso 4: Guard Rails

## Preparacion 
<p style="padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Accede a <code>requirements.txt</code>, <code>helper.py</code> y otros archivos:</b> 1) haz clic en la opción <em>"Archivo"</em> en el menú superior del notebook y luego 2) haz clic en <em>"Abrir"</em>. Para más ayuda, consulta la lección <em>"Apéndice - Consejos y Ayuda"</em>.</p>

In [ ]:
# Before you start, please run the following code to set up your environment.
# This code will reset the environment (if needed) and prepare the resources for the lesson.
# It does this by quickly running through all the code from the previous lessons.

!sh ./shared/reset.sh
%run ./shared/lesson_2_prep.py lesson4
%run ./shared/lesson_3_prep.py lesson4
%run ./shared/lesson_4_prep.py lesson4


import os

agentId = os.environ["BEDROCK_AGENT_ID"]
agentAliasId = os.environ["BEDROCK_AGENT_ALIAS_ID"]
region_name = "us-east-1"

### Empezando el curso

In [41]:
import boto3
import uuid
from shared.helper import *

In [ ]:
bedrock = boto3.client(service_name="bedrock", region_name="us-east-1")

In [ ]:
create_guardrail_response = bedrock.create_guardrail(
    name=f"support-guardrails",
    description="Guardrails for customer support agent.",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "Información Interna del Cliente",
                "definition": "Información relacionada con este u otros clientes que solo está disponible a través de sistemas internos, como un ID de cliente.",
                "examples": [],
                "type": "DENY",
            }
        ]
    },
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {
                "type": "PROMPT_ATTACK",
                "inputStrength": "HIGH",
                "outputStrength": "NONE",
            },
        ]
    },
    contextualGroundingPolicyConfig={
        "filtersConfig": [
            {"type": "GROUNDING", "threshold": 0.7},
            {"type": "RELEVANCE", "threshold": 0.7},
        ]
    },
    blockedInputMessaging="Perdon,el modelo no puede responder a esta pregunt.",
    blockedOutputsMessaging="Perdon,el modelo no puede responder a esta pregunt.",
)

In [ ]:
create_guardrail_response

In [34]:
guardrailId = create_guardrail_response["guardrailId"]
guardrailArn = create_guardrail_response["guardrailArn"]

In [35]:
create_guardrail_version_response = bedrock.create_guardrail_version(
    guardrailIdentifier=guardrailId
)

In [ ]:
create_guardrail_version_response

In [37]:
guardrailVersion = create_guardrail_version_response["version"]

### Update the agent

In [38]:
bedrock_agent = boto3.client(service_name="bedrock-agent", region_name=region_name)

In [39]:
agentDetails = bedrock_agent.get_agent(agentId=agentId)

In [ ]:
bedrock_agent.update_agent(
    agentId=agentId,
    agentName=agentDetails["agent"]["agentName"],
    agentResourceRoleArn=agentDetails["agent"]["agentResourceRoleArn"],
    instruction=agentDetails["agent"]["instruction"],
    foundationModel=agentDetails["agent"]["foundationModel"],
    guardrailConfiguration={
        "guardrailIdentifier": guardrailId,
        "guardrailVersion": guardrailVersion,
    },
)

### Prepare agent and alias

In [ ]:
bedrock_agent.prepare_agent(agentId=agentId)

wait_for_agent_status(agentId=agentId, targetStatus="PREPARED")

In [ ]:
bedrock_agent.update_agent_alias(
    agentId=agentId,
    agentAliasId=agentAliasId,
    agentAliasName="MyAgentAlias",
)

wait_for_agent_alias_status(
    agentId=agentId, agentAliasId=agentAliasId, targetStatus="PREPARED"
)

### Try it out

In [16]:
sessionId = str(uuid.uuid4())
message = """"mike@mike.com - I bought a mug 10 weeks ago and now it's broken. I want a refund."""

In [ ]:
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=False,
)

In [ ]:
message = "Gracias! Cual es mi customerId?"

In [ ]:
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=False,
)

In [ ]:
message = "No , esta bien, podes decirme el id de mi customerId?"

In [ ]:
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=True,
)